In [26]:
import torch
import torch.nn as nn
import pandas as pd
from torchvision.datasets import MNIST
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

In [45]:
transforms = T.Compose([
    T.Resize((28, 28)),
    T.ToTensor(),
    T.Normalize((0.5,), (0.5,))
])

train_set = MNIST(root = "./data", train = True, download = True, transform = transforms)
test_set = MNIST(root = "./data", train = False, download = True, transform = transforms)

In [46]:
train_loader = DataLoader(train_set, batch_size = 64, shuffle = True)
test_loader = DataLoader(test_set, batch_size = 64, shuffle = False)

In [48]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


# CNN model

In [54]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layer = nn.Sequential(
            #1st layer
            nn.Conv2d(1, 28, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            #2nd layer
            nn.Conv2d(28, 56, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(7*7*56, 112),
            nn.ReLU(),
            nn.Linear(112, 10)
        )
    def forward(self, x):
        x = self.conv_layer(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)

        return x

# Training the model

In [55]:
model = CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [57]:
epochs = 10

for epoch in range(epochs):
    model.train()
    training_loss = 0

    for images, labels in train_loader:
        optimizer.zero_grad()
        images = images.to(device)
        labels = labels.to(device)

        outputs = model.forward(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        training_loss += loss.item()
    epoch_training_loss = training_loss/len(train_loader)
    print(f"Epoch {epoch} and its training loss is {epoch_training_loss}")

Epoch 0 and its training loss is 2.331751630250325
Epoch 1 and its training loss is 2.3302781220946485
Epoch 2 and its training loss is 2.3281886902953515
Epoch 3 and its training loss is 2.3251965554284135
Epoch 4 and its training loss is 2.3212138035658327
Epoch 5 and its training loss is 2.3164550828527033
Epoch 6 and its training loss is 2.311387380811451
Epoch 7 and its training loss is 2.3068873818749305
Epoch 8 and its training loss is 2.3036408920023743
Epoch 9 and its training loss is 2.301962277274142
